In [1]:
import pandas as pd
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig,  BitsAndBytesConfig


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [30]:
from tqdm import tqdm
from datasets import Dataset
from torch.utils.data import DataLoader

batchsize = 32

def embedd_and_save_evidence(evidence_df, embedd_test_path):
    evidence_embeddings = []
    
    #prepare data
    evidence_ds = Dataset.from_pandas(evidence_df)
    evidence_dl = DataLoader(evidence_ds, batch_size=batchsize, shuffle=False)
    for evidence in tqdm(evidence_dl):
        evidence_embeddings.append(model.encode(evidence['text']).cpu().detach().numpy())
        
    # turn to np arr
    evidence_embeddings = np.concatenate(evidence_embeddings, axis=0)
    print(evidence_embeddings.shape)
    np.save(embedd_test_path, evidence_embeddings)

In [33]:
# read evidence data
evidence_test_path = '/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#dump directory
embedd_test_path = '/raid/deallab/SF_RAG_Data/ASQA/test/embedd_test.npy'
embedd_and_save_evidence(evidence_df, embedd_test_path)

  0%|          | 0/365 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/0783263f3c009f67bd0e177040cfecad4b1171d6/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/raid/deallab/anaconda3/envs/lasse_sf_rag/lib/python3.11/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
100%|██████████| 365/365 [30:47<00:00,  5.06s/it]

(11657, 4096)
